# 🐍 Python Bootcamp 2026 — Top-Down Demo Notebook
### Stevens School of Business · Information Systems Area · Saturday, Oct 10, 2026

**How to use this notebook**

* Click inside a grey *code cell* and press **Shift + Enter** to run it. The result appears underneath.
* Run cells **top to bottom**. If something breaks, *Run → Run All Above*, or restart from the top.
* Cells marked **🧪 Try it** are for you. Change a value, re-run, see what happens. That loop *is* programming.
* You can't break anything. Worst case: *Kernel → Restart* and run again.

**The dataset: Superstore.** Four years (2015–2018) of orders from a US office-supplies retailer — 9,994 order lines,
793 customers, 4 regions, 3 product categories. It's the sample dataset that ships with Tableau and the one you'll
see in a thousand analytics tutorials and interview take-homes. Today you'll build a real dashboard on it.

> **Setup check:** if the next cell errors with `No module named pandas`, run `%pip install pandas matplotlib`
> in a new cell, then restart the kernel.

## Part 0 — Start at the finish line (9:30 demo)

Here's the *entire* business logic of today's dashboard. Run it. Don't try to understand it yet — that's what the next two hours are for.

In [ ]:
import pandas as pd

orders = pd.read_csv("data/superstore.csv")

total_sales   = orders["Sales"].sum()
total_profit  = orders["Profit"].sum()
order_count   = orders["Order ID"].nunique()
best_category = orders.groupby("Category")["Profit"].sum().idxmax()

print(f"Total sales:    ${total_sales:,.0f}")
print(f"Total profit:   ${total_profit:,.0f}  ({total_profit / total_sales:.1%} margin)")
print(f"Orders:         {order_count:,}")
print(f"Most profitable category: {best_category}")

Four lines of math on a spreadsheet → four KPI tiles on a dashboard. That's the whole trick.
By the end of Part 2 you'll be able to read every line above.

---
## Part 1 — Variables, types, and printing (10:30 – 11:30)

### 1.1 A variable is a labeled box
Here's the very first row of the Superstore file, as individual variables.

In [ ]:
product  = "Bush Somerset Collection Bookcase"
quantity = 2
sales    = 261.96
profit   = 41.91
returned = False

print(product)
print(quantity)

### 1.2 Every value has a type
Python keeps track of what kind of thing is in each box. `type()` tells you.

In [ ]:
print(type(product))    # str   → text ("string")
print(type(quantity))   # int   → whole number
print(type(sales))      # float → decimal number
print(type(returned))   # bool  → True / False

Why care? **Types decide what you're allowed to do.** Numbers divide; text doesn't.

In [ ]:
unit_price = sales / quantity
margin = profit / sales
print(unit_price)
print(margin)

### 1.3 Your first bug (on purpose)
Run the next cell. It **will** error — that's the point. Scroll to the **last line** of the red text. That line is the one that matters.

In [ ]:
label = "Profit: " + profit    # ← trying to glue text onto a number
print(label)

`TypeError: can only concatenate str (not "float") to str` → Python is saying *"you gave me text and a number and said 'add'. Which did you mean?"*

**Fix:** convert the number to text, or (better) use an *f-string*.

> 🤖 **AI-assistant habit #1:** paste the *last line* of any error into Copilot Chat / ChatGPT / Claude and ask
> *"What does this mean and how do I fix it?"* You'll do this 20 times today. That's normal.

In [ ]:
label = "Profit: " + str(profit)    # fix 1: convert with str()
print(label)

print(f"Profit: {profit}")          # fix 2: f-string — f before the quote, variables in {braces}

### 1.4 f-strings: how dashboards format numbers
After a colon inside the braces you can add a *format spec*. These three cover 90% of BI:

In [ ]:
big_number = 2297200.8603

print(f"{big_number}")          # raw
print(f"{big_number:,.0f}")     # thousands separator, 0 decimals → 2,297,201
print(f"${big_number:,.2f}")    # currency                        → $2,297,200.86
print(f"{0.1247:.1%}")          # percent, 1 decimal               → 12.5%

### 1.5 🧪 Try it — a KPI by hand
Replace the `...` so the cell prints a profit margin as a percentage. *Hint: margin = profit ÷ sales.*
This is the second row of Superstore: three chairs.

In [ ]:
sales = 731.94
profit = 219.58

margin = ...            # ← profit divided by sales

print(f"Sales:  ${sales:,.2f}")
print(f"Margin: {margin:.1%}")

<details><summary>✅ Answer (click)</summary>

```python
margin = profit / sales      # → 30.0%
```
</details>

### 1.6 Comments
Anything after `#` is ignored. Use comments to explain *why*, not *what*.

In [ ]:
TARGET_MARGIN = 0.15    # the CFO's target; change here if it moves
chair_margin = 219.58 / 731.94
print(f"Chairs margin {chair_margin:.1%} — above target: {chair_margin > TARGET_MARGIN}")

---
## Part 2 — Lists, dictionaries, and your first look at pandas (11:30 – 12:30)

### 2.1 A list is a column
Ordered. Square brackets. Counting starts at **0**.

In [ ]:
regions = ["West", "East", "Central", "South"]

print(regions[0])        # first
print(regions[-1])       # last
print(len(regions))      # how many

regions.append("International")
print(regions)

### 2.2 Loops: do something to every item

In [ ]:
yearly_sales = [484247, 470532, 609206, 733215]     # 2015, 2016, 2017, 2018

for amount in yearly_sales:
    print(f"${amount:,}")

print("Total:", f"${sum(yearly_sales):,}")
print("Best year:", f"${max(yearly_sales):,}")

### 2.3 A dictionary is a row
Keys → values. Curly braces. One spreadsheet row with the column names attached.

In [ ]:
line = {
    "Order ID": "CA-2017-152156",
    "Region": "South",
    "Category": "Furniture",
    "Product Name": "Bush Somerset Collection Bookcase",
    "Sales": 261.96,
    "Profit": 41.91,
}

print(line["Product Name"])                 # look up by key
print(line["Profit"] / line["Sales"])       # compute from fields

line["Returned"] = False                    # add a column to this row
print(line)

### 2.4 A list of dictionaries is… a table

In [ ]:
lines = [
    {"Region": "South", "Category": "Furniture",       "Sales": 261.96},
    {"Region": "South", "Category": "Furniture",       "Sales": 731.94},
    {"Region": "West",  "Category": "Office Supplies", "Sales":  14.62},
]

total = 0
for row in lines:
    if row["Category"] == "Furniture":
        total = total + row["Sales"]

print(f"Furniture sales: ${total:,.2f}")

That works — but it's 5 lines for one number, and the real file has 9,994 rows and 21 columns.
**This is exactly the problem pandas solves.** A *DataFrame* is a list-of-dicts with superpowers.

In [ ]:
import pandas as pd

df = pd.DataFrame(lines)    # the same 3 rows, as a DataFrame
df

In [ ]:
# The 5-line loop above becomes one line:
df[df["Category"] == "Furniture"]["Sales"].sum()

### 2.5 Load the real dataset

In [ ]:
orders = pd.read_csv("data/superstore.csv")
orders.head()          # first 5 rows

**The "get to know your data" trio.** Run these on *any* new dataset, always, before anything else.

In [ ]:
print(orders.shape)        # (rows, columns)
print()
print(orders.columns.tolist())

In [ ]:
orders.info()              # column types + missing values

In [ ]:
orders.describe()          # summary stats for every numeric column

> **Two things to notice.** Column names have **spaces** (`"Order Date"`, `"Sub-Category"`) — that's normal in real
> data; just always use the `orders["Column Name"]` bracket form. And `describe()` shows **Profit has a minimum of
> −$6,600** — some orders *lose* money. Hold that thought.

### 2.6 Pick columns, filter rows

In [ ]:
orders["Sales"]                         # one column (a "Series")

In [ ]:
orders[["Order Date", "Region", "Category", "Product Name", "Sales", "Profit"]].head()    # several → double brackets

In [ ]:
big_orders = orders[orders["Sales"] > 1000]       # keep rows where the condition is True
print(len(big_orders), "order lines over $1,000")
big_orders.head()

In [ ]:
# Combine conditions with & (and) / | (or). Each condition in its own parentheses.
losses = orders[(orders["Profit"] < 0) & (orders["Discount"] >= 0.5)]
print(len(losses), "lines lost money AND had a 50%+ discount")

### 2.7 KPIs: the numbers that go on the tiles

One subtlety that trips up real analysts: each **row is an order *line*** (one product in one order). A single order
can have several lines. So "number of orders" is the number of *distinct* Order IDs, not the number of rows.

In [ ]:
total_sales  = orders["Sales"].sum()
total_profit = orders["Profit"].sum()
margin       = total_profit / total_sales
line_count   = len(orders)
order_count  = orders["Order ID"].nunique()        # distinct orders
avg_order    = total_sales / order_count
avg_discount = orders["Discount"].mean()
customers    = orders["Customer ID"].nunique()

print(f"Sales:          ${total_sales:,.0f}")
print(f"Profit:         ${total_profit:,.0f}")
print(f"Margin:         {margin:.1%}")
print(f"Order lines:    {line_count:,}")
print(f"Orders:         {order_count:,}")
print(f"Avg order:      ${avg_order:,.2f}")
print(f"Avg discount:   {avg_discount:.1%}")
print(f"Customers:      {customers:,}")

### 2.8 `groupby` — the pivot table of Python
"Sales **by** region" → `groupby("Region")`. This one method is most of BI.

In [ ]:
orders.groupby("Region")["Sales"].sum()

In [ ]:
# Sort it, round it
by_region = (
    orders.groupby("Region")["Sales"]
          .sum()
          .sort_values(ascending=False)
          .round(0)
)
by_region

In [ ]:
# Several metrics at once — and here's the hidden story in this dataset
by_category = orders.groupby("Category").agg(
    sales=("Sales", "sum"),
    profit=("Profit", "sum"),
    orders=("Order ID", "nunique"),
).round(0)
by_category["margin"] = (by_category["profit"] / by_category["sales"]).round(3)
by_category.sort_values("sales", ascending=False)

Furniture sells almost as much as Technology — and makes **one-eighth** the profit. A dashboard that only shows sales
would hide that completely. Let's go one level down:

In [ ]:
by_subcat = orders.groupby("Sub-Category")["Profit"].sum().sort_values()
by_subcat.round(0)

**Tables lose $17,725. Bookcases lose $3,473.** That's the slide a BI analyst shows the CFO. One `groupby`.

In [ ]:
# Why? Discounts. Compare margin on discounted vs. full-price lines:
orders["Discounted"] = orders["Discount"] > 0
disc = orders.groupby("Discounted").agg(sales=("Sales", "sum"), profit=("Profit", "sum"))
disc["margin"] = (disc["profit"] / disc["sales"]).round(3)
disc

In [ ]:
# Counting things: value_counts()
orders["Ship Mode"].value_counts()

### 2.9 Dates → trends over time
Right now `Order Date` is just text like `"11/8/2017"`. Tell pandas it's a date and you unlock month/year math.

In [ ]:
orders["Order Date"] = pd.to_datetime(orders["Order Date"], format="%m/%d/%Y")
orders["Year"]  = orders["Order Date"].dt.year
orders["Month"] = orders["Order Date"].dt.to_period("M")      # 2017-11, 2017-12, ...

orders.groupby("Year")["Sales"].sum().round(0)

In [ ]:
monthly = orders.groupby("Month")["Sales"].sum()
monthly.tail(12)

### 2.10 A chart in one line

In [ ]:
monthly.plot(kind="line", title="Monthly sales, 2015–2018", figsize=(10, 3));

In [ ]:
by_subcat.plot(kind="barh", title="Profit by sub-category", figsize=(8, 5), color="#A32638");

### 2.11 🧪 Try it — answer three business questions
Each is one line using what's above. Replace the `...`.

In [ ]:
# Q1: Total sales for the "Home Office" segment
q1 = ...

# Q2: Which State has the highest total sales?  (hint: groupby("State")["Sales"].sum().idxmax())
q2 = ...

# Q3: Average discount by Region
q3 = ...

print(q1); print(q2); print(q3)

<details><summary>✅ Answers (click)</summary>

```python
q1 = orders[orders["Segment"] == "Home Office"]["Sales"].sum()          # 429,653
q2 = orders.groupby("State")["Sales"].sum().idxmax()                     # California
q3 = orders.groupby("Region")["Discount"].mean()                         # Central is highest
```
</details>

### 2.12 Look back at Part 0
Scroll up to the very first code cell. You can now read every line of it. 🎉

---
## Part 3 — The bridge to Streamlit (1:15 PM, after lunch)

Everything you just did maps 1-to-1 onto a dashboard. Streamlit turns a plain Python *script* (`app.py`, not a notebook) into a web page:

| In this notebook | In `app.py` (Streamlit) |
|---|---|
| `print(f"Sales: ${total_sales:,.0f}")` | `st.metric("Sales", f"${total_sales:,.0f}")` |
| `orders.head()` | `st.dataframe(orders)` |
| `by_region.plot(kind="bar")` | `st.bar_chart(by_region)` |
| `monthly.plot()` | `st.line_chart(monthly)` |
| `orders[orders["Region"] == "West"]` | `region = st.sidebar.multiselect(...)` then `.isin(region)` |
| markdown cell | `st.title(...)`, `st.subheader(...)` |

Run the app from the terminal: `streamlit run app.py`

The next cell isn't runnable here — it's the *minimum* Streamlit app, shown so you recognize it when the AI assistant writes it for you.

In [ ]:
# checkpoints/app_v1.py — the smallest possible dashboard (save as a .py file; don't run in the notebook)
#
# import streamlit as st
# import pandas as pd
#
# st.title("Superstore Sales Dashboard")
# orders = pd.read_csv("data/superstore.csv")
# st.metric("Total sales", f"${orders['Sales'].sum():,.0f}")
# st.dataframe(orders)

print("See checkpoints/app_v1.py in the starter folder.")

---
## Part 4 — Working with an AI assistant (any of them)

**With GitHub Copilot Chat:** open the chat panel (`Ctrl/⌘ + Alt + I` in VS Code), keep `app.py` open so it has context, paste a prompt.
**Without Copilot:** open ChatGPT / Claude / Gemini in a browser tab next to VS Code. Same prompts — paste your code *and* the error in, copy the answer back out. The skill is identical.

### The loop you'll repeat all afternoon
1. **Prompt** — say what you want *and* what you already have (paste your current `app.py`).
2. **Read** — skim before pasting. Do you recognize `read_csv`, `groupby`, `st.metric`? You should by now.
3. **Run** — `streamlit run app.py`. Look at the browser.
4. **Fix** — error? Paste the last 3 lines back into the chat. Ugly? Say what's wrong in plain English.

### Prompts for today (also in `PROMPTS.md`)

**Scaffold (1:15):**
> I'm a beginner building a Streamlit dashboard. I have `data/superstore.csv` with columns: Row ID, Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit. Column names contain spaces. Write a complete `app.py` that loads the CSV with pandas, shows a title, four KPI metrics in a row of columns (total sales, total profit, number of distinct orders using `Order ID`, average order value = sales ÷ orders), and the raw data in a table. Format money with $ and commas. Only use streamlit and pandas. Add a short comment above each section.

**Filters + charts (2:15):**
> Here is my current `app.py`: [paste]. Add a sidebar with multiselects for `Region` and `Category`, defaulting to all values. Every KPI, chart and table must use the filtered data. Then add a line chart of monthly sales (parse `Order Date` with format "%m/%d/%Y" first) and a horizontal bar chart of profit by `Sub-Category`, sorted so the losses are visible.

**Debug (any time):**
> I ran `streamlit run app.py` and got this error: [paste last 3 lines]. Here is my code: [paste]. Explain the cause in one sentence, then give me the fixed file.

**Polish (2:30):**
> Make this dashboard look professional: `st.set_page_config` with a wide layout and page title; a caption with the row count after filtering; a date-range filter; a profit-margin delta on the profit metric; a "Top 10 products by sales" table; a download button for the filtered data as CSV; a friendly warning if the filters return no rows.

### Three rules
* **Never paste code you haven't read.** Mystery line? Ask: *"explain line 12."*
* **One change at a time.** Big prompts → big errors.
* **The AI is wrong ~1 in 5 times.** The error message tells you which time.

---
## Appendix — Cheat sheet

```python
# Load
df = pd.read_csv("file.csv")

# Inspect
df.head()  df.shape  df.columns  df.info()  df.describe()

# Select
df["Sales"]                  df[["Region", "Sales"]]

# Filter
df[df["Sales"] > 1000]       df[(df["Profit"] < 0) & (df["Discount"] >= 0.5)]

# Aggregate
df["Sales"].sum() / .mean() / .max() / .count()     df["Order ID"].nunique()
df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
df.groupby("Category").agg(sales=("Sales", "sum"), profit=("Profit", "sum"))
df["Ship Mode"].value_counts()

# Dates
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Order Date"].dt.year        df["Order Date"].dt.to_period("M")

# Format
f"${x:,.0f}"   f"{x:.1%}"   f"{x:,.2f}"
```

**Dataset:** Tableau "Sample – Superstore" (Orders), 9,994 rows, 2015–2018, cleaned to UTF-8. Widely redistributed as a teaching dataset; see `data/README.md`.

*Stevens Python Bootcamp 2026 · Information Systems Area · Facilitators: Uday Adusumilli & Foad Mahdavi Pajouh*